# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 04b Corrected P5 test evaluation

**Permitted partitions:** TRAIN + VALIDATION for preprocessing identity and frozen-model prediction replay; then FINAL TEST — READ ONLY after every replay passes. No model training, tuning or model selection is performed.

This corrects the documented 2026-10-12 restored-feature-order defect. The original `p5-clean-01` artifacts and Google Drive models remain untouched. Use a fresh CPU runtime, the same original Drive directory and a new correction run ID. Allow time for downloading the original evidence and replaying all models; this does not repeat 54 tuning fits. Only one Colab session may run this recovery. See `docs/P5_RECOVERY.md`.

## 1. Bootstrap
Use the code SHA from the same GitHub revision as this notebook. Installing the declared environment can require a kernel restart; preserve the checkout and rerun setup.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = (os.environ.get("AIRSENSE_CODE_REVISION") or input("Code version: full 40-character Git commit SHA (not a run ID): ")).strip().lower()
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p5.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)


## 2. Provenance and GitHub authentication
New correction run: `p5-clean-01-corrected-02`. Reuse this ID only to resume this correction at the same code revision/environment. Never enter `p5-clean-01` here.

In [ ]:
import logging
from airsense_r.artifacts import provenance, json_bytes
from airsense_r.artifacts import source_hash
from airsense_r.checkpoints import open_run
from airsense_r.colab import configure_github_secret

logging.basicConfig(level=logging.INFO)
if source_hash(ROOT) != "81627e5dc01a7d3f6427b7fc8d74403a21338dc887b1c2abc1475fc408169dcd":
    raise RuntimeError("Notebook and scientific source revisions differ; open the matching notebook revision")
if TEST_MODE:
    RUN_ID = os.environ["AIRSENSE_NOTEBOOK_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_REMOTE")
else:
    from google.colab import userdata
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    RUN_ID = input("Run name, e.g. p5-clean-01-corrected-02 (same name only for resume; not the code SHA): ").strip()
    REMOTE = REPOSITORY


## 3. Original persistent model storage
Use `/content/drive/MyDrive/AirSense-R/model-checkpoints`, the original parent directory. The script reads the existing `p5-clean-01` model subdirectory without modifying it.

In [ ]:
STAGE = "corrected-test"
storage_text = input("Persistent model directory (e.g. /content/drive/MyDrive/AirSense-R/model-checkpoints): ").strip()
if not storage_text:
    raise ValueError("A persistent model directory is required")
MODEL_DIRECTORY = Path(storage_text).expanduser().resolve()
if str(MODEL_DIRECTORY).startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not mounted")
elif str(MODEL_DIRECTORY).startswith("/content/"):
    raise ValueError("Ordinary /content storage is ephemeral; use mounted persistent storage")
print("Stage:", STAGE)
print("Model storage:", MODEL_DIRECTORY)


## 4. Verify development predictions, then rescore
Output is streamed below. Each passed model replay and completed test shard is backed up to the correction result branch. The script locks out a second recovery in this runtime. If any check fails, stop and share the error; do not change tolerances or retrain. A runtime interruption may require repeating prediction work, but never model fitting.

In [ ]:
command = [sys.executable, str(ROOT / "scripts/recover_p5_test.py"),
           "--run-id", RUN_ID, "--model-directory", str(MODEL_DIRECTORY), "--sync-remote", REMOTE]
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"}) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if return_code:
    raise RuntimeError(f"Recovery stopped ({return_code}). Read the error above; do not start another copy.")


## 5. Verify and display corrected evidence
Run after cell 4 succeeds. Save this executed notebook separately from the original run.

In [ ]:
import json
import pandas as pd
from airsense_r.artifacts import RunStore
from airsense_r.p5 import require_selection
from IPython.display import Image
output = ROOT / ".tmp/checkpoint-checkouts" / RUN_ID / "results" / RUN_ID
manifest = json.loads((output / "manifest.json").read_text())
store = RunStore(output, manifest["provenance"])
decision = require_selection(store)
print("Frozen selected family:", decision["selected_family"])
display(store.load_json("validation-replay-complete.json"))
display(store.load_json("complete.json"))
display(pd.read_csv(output / "metrics-test-aggregate.csv"))
display(store.load_json("test-uncertainty.json"))
display(Image(filename=str(output / "figure-test-mae.png")))


## Interpretation
The original test report is invalidated; retain it as incident evidence. Corrected results must cite the correction lineage and post-results disclosure. Keep models and selection frozen. Send the executed recovery notebook for review before P6.